In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import sklearn.model_selection 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

#завантаження набору даних про дощі в Австралії
dataset_path = kagglehub.dataset_download("jsphyg/weather-dataset-rattle-package")
weather_data = pd.read_csv(dataset_path +'/weatherAUS.csv')
print(weather_data.head())
print(weather_data.info())

#видалення пропусків та колонок
missing_value_ratio = weather_data.isnull().mean()
columns_remove = missing_value_ratio[missing_value_ratio > 0.5].index
weather_data = weather_data.drop(columns= columns_remove)

#зміна типу колонок, створення додаткових
weather_data['Date'] = pd.to_datetime(weather_data['Date'], errors= 'coerce')
weather_data['Year'] = weather_data['Date'].dt.year
weather_data['Month'] = weather_data['Date'].dt.month

weather_data = weather_data.dropna(subset= ['RainTomorrow'])
weather_data['RainTomorrow'] = weather_data['RainTomorrow'].map({'Yes' : 1, 'No': 0})

#розділення на числові та категоріальні ознаки
target_columns = 'RainTomorrow'
numeric_columns = weather_data.select_dtypes(include=['int64','float64']).columns.tolist()
numeric_columns = [col for col in numeric_columns if col != target_columns]
category_columns = []
category_columns = [col for col in category_columns if col != 'RainTomorrow']

#надалі рік числовий, місяць категоріальний
if 'Year' in category_columns:
    category_columns.remove('Year')

if 'Month' in numeric_columns:
    numeric_columns.remove('Month')
category_columns.append('Month')

#тренувальна та тестова вибірки
max_year = weather_data['Year'].max()
train_data = weather_data[weather_data['Year'] < max_year]
test_data = weather_data[weather_data['Year'] == max_year]

x_train = train_data[numeric_columns + category_columns]
y_train = train_data[target_columns]

x_test = test_data[numeric_columns + category_columns]
y_test = test_data[target_columns]

#відновлення пропущених даних
numeric_imputer = SimpleImputer(strategy= 'median')
category_imputer = SimpleImputer(strategy= 'most_frequent')

x_train_numeric = pd.DataFrame(numeric_imputer.fit_transform(x_train[numeric_columns]), columns= numeric_columns)
x_test_numeric = pd.DataFrame(numeric_imputer.transform(x_test[numeric_columns]), columns= numeric_columns)

x_train_category = pd.DataFrame(category_imputer.fit_transform(x_train[category_columns]), columns= category_columns)
x_test_category = pd.DataFrame(category_imputer.transform(x_test[category_columns]), columns= category_columns)

#стандартизація 
feature_scaler = StandardScaler()

x_train_numeric = feature_scaler.fit_transform(x_train_numeric)
x_test_numeric = feature_scaler.transform(x_test_numeric)

#кодування категоріальних ознак
one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

x_train_category = one_hot_encoder.fit_transform(x_train_category)
x_test_category = one_hot_encoder.transform(x_test_category)

#обʼєднання підмножини з числовими і категоріальними ознаками
x_train_fin = np.hstack([x_train_numeric, x_train_category])
x_test_fin = np.hstack([x_test_numeric, x_test_category])

logistic_model = LogisticRegression(max_iter= 200)
logistic_model.fit(x_train_fin, y_train)

predictions = logistic_model.predict(x_test_fin)

#результат
print(classification_report(y_test, predictions))

Using Colab cache for faster access to the 'weather-dataset-rattle-package' dataset.
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   
2         WSW           46.0          W  ...        38.0         30.0   
3          NE           24.0         SE  ...        45.0         16.0   
4           W           41.0        ENE  ...        82.0         33.0   

   Pressure9a